# Demo 2 — El mismo archivo, dos prompts (OpenAI)

**Se graba en el capítulo 3, ~35 segundos en pantalla, sin cortes.**

Orden: 0 → 1 → 2 (prompt vago) → 3 (prompt estructurado) → 4 (comparación).
Ejecuta todo antes de grabar y luego reinicia el kernel: en cámara solo se ven las salidas nuevas.

Requisitos: `pip install -r requirements.txt` (openai, pandas, tabulate, tiktoken) y la clave en la variable de entorno `OPENAI_API_KEY`.

## 0. Setup

La celda de abajo lista los modelos de tu cuenta. **Copia el ID exacto** que quieras usar en
`MODELO`: los nombres cambian cada pocos meses y un ID inventado peta en mitad de la toma.

In [ ]:
import os, pandas as pd
import openai
from openai import OpenAI

assert os.environ.get("OPENAI_API_KEY"), "Falta la variable de entorno OPENAI_API_KEY"
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

disponibles = sorted(m.id for m in client.models.list() if m.id.startswith("gpt-"))
print("\n".join(disponibles))

In [ ]:
MODELO = "gpt-5.6-terra"   # ← pega aquí el ID exacto de la lista de arriba

def preguntar(prompt, temperatura=0):
    """Responses API, con dos caídas suaves: modelos razonadores que rechazan
    `temperature` (el servidor devuelve un 400) y SDKs antiguos que aún no tienen `.responses`."""
    try:
        r = client.responses.create(model=MODELO, input=prompt, temperature=temperatura)
        return r.output_text
    except openai.BadRequestError as e:
        if "temperature" not in str(e):
            raise
        r = client.responses.create(model=MODELO, input=prompt)   # razonadores: sin temperature
        return r.output_text
    except AttributeError:
        r = client.chat.completions.create(model=MODELO, temperature=temperatura,
                                           messages=[{"role": "user", "content": prompt}])
        return r.choices[0].message.content

print("Probando:", preguntar("Responde solo con la palabra OK."))

## 1. Lo que hay en el archivo

Para la locución: hay **nulos** en `precio_unitario`, `ingresos` y `unidades`. Es a propósito.
Es lo que va a separar los dos prompts.

In [ ]:
df = pd.read_csv("ventas_trimestre.csv")

print(f"{len(df)} filas · {df.mes.nunique()} meses · {df.producto.nunique()} productos\n")
print("Valores que faltan:")
print(df.isna().sum()[lambda s: s > 0].to_string())
df.head()

## 2. EL PROMPT VAGO ❌

Una línea, el CSV entero pegado a cholón, y a ver qué sale.
Fíjate en el tamaño: se está mandando el archivo completo porque *cabe*.

In [ ]:
prompt_malo = f"""Analízame este CSV de ventas y dime qué está pasando.

{df.to_csv(index=False)}"""

print(f"Tamaño del prompt: {len(prompt_malo):,} caracteres\n")
respuesta_mala = preguntar(prompt_malo)
print(respuesta_mala)

### Qué señalar en pantalla

- Resumen genérico que vale para **cualquier** archivo de ventas
- No dice nada de los nulos: los ha ignorado o los ha estimado sin avisar
- No compara trimestres, que era lo único que importaba
- Y ha costado mandar el archivo entero

## 3. EL PROMPT ESTRUCTURADO ✅

Seis bloques. Y en vez del CSV entero, **solo el agregado que hace falta**: menos tokens,
menos ruido y mejor respuesta.

In [ ]:
# Primero se decide qué es relevante. Eso lo decide una persona, no el modelo.
df["trimestre"] = df.mes.str.startswith("2025").map({True: "Q4_2025", False: "Q1_2026"})
tabla = (df.pivot_table(index="producto", columns="trimestre",
                        values="ingresos", aggfunc="sum")
           .round(0))
tabla["var_%"] = ((tabla.Q1_2026 / tabla.Q4_2025 - 1) * 100).round(1)
tabla = tabla.sort_values("var_%")

nulos = df.isna().sum()[lambda s: s > 0].to_dict()
tabla

In [ ]:
prompt_bueno = f"""ROL
Eres analista de datos con diez años de experiencia en retail de electrónica de consumo.

OBJETIVO
Identificar los tres productos con mayor caída de ingresos entre Q4 2025 y Q1 2026,
y proponer para cada uno la explicación más probable.

INSTRUCCIONES
1. Ordena los productos por variación porcentual de ingresos.
2. Quédate con los tres que más caen.
3. Para cada uno, propón una hipótesis y di qué dato haría falta para confirmarla.

RESTRICCIONES
- No inventes datos que no estén en la tabla.
- Q4 incluye la campaña de navidad, así que la comparación no es directa: dilo si afecta.
- Estos valores faltan en el archivo original: {nulos}. Menciónalo si condiciona la conclusión.
- No estimes lo que falte. Si un dato no está, dilo.

FORMATO DE SALIDA
Una tabla markdown con: producto | variación % | hipótesis | dato que falta.
Debajo, máximo tres líneas de conclusión. Sin introducción ni despedida.

DATOS
{tabla.to_markdown()}"""

print(f"Tamaño del prompt: {len(prompt_bueno):,} caracteres\n")
respuesta_buena = preguntar(prompt_bueno)
print(respuesta_buena)

### Qué señalar en pantalla

- Responde exactamente lo que se le pidió, en el formato que se le pidió
- Avisa del efecto navidad porque se le dijo que lo tuviera en cuenta
- Menciona los nulos en vez de tragárselos
- Y ha usado **muchísimos menos tokens**

## 4. El remate: menos contexto, mejor respuesta

Aquí se puede contar en tokens de verdad, con el mismo `tiktoken` del notebook 1.

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("o200k_base")

t_malo, t_bueno = len(enc.encode(prompt_malo)), len(enc.encode(prompt_bueno))

print(f"Prompt vago         {t_malo:>7,} tokens")
print(f"Prompt estructurado {t_bueno:>7,} tokens")
print(f"\nEl bueno usa un {1 - t_bueno/t_malo:.0%} menos de contexto... y responde mejor.")

---
### Chuleta para la locución

> "No es que la máquina se haya vuelto más lista de repente. Es que las decisiones
> las ha tomado una persona: qué es relevante, qué formato hace falta y qué no se puede inventar."

**Respuesta correcta esperada** (para comprobar que el modelo no se inventa nada):
Robot aspirador R2 (−42 %), Auriculares BT Zen (−38 %), Altavoz portátil Mini (−33 %).

### Si falla la red en mitad de la grabación
Guarda las dos respuestas la víspera:
```python
open("salida_mala.txt",  "w").write(respuesta_mala)
open("salida_buena.txt", "w").write(respuesta_buena)
```
Y en la toma, léelas del disco en vez de llamar a la API.